# Makine Öğrenmesi Ara Ödevi
Müşteri Ayrılma Tahmini ile Temel Makine Öğrenmesi Akışı

## 1. Docstring ve kütüphaneler

In [ ]:
"""Amaç: Sentetik müşteri verileriyle churn sınıflandırması yapmak.
Kütüphaneler: numpy, pandas, matplotlib, scikit-learn.
Çalıştırma: Hücreleri yukarıdan aşağıya çalıştırın."""
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,ConfusionMatrixDisplay
from sklearn.base import clone
RANDOM_STATE=42

## 2. Sentetik veri setini oluşturma

In [ ]:
rng=np.random.default_rng(RANDOM_STATE); n=500
yas=rng.integers(18,71,n); gelir=np.clip(rng.normal(52000,18000,n),15000,120000).round()
abonelik_suresi=rng.integers(1,121,n); destek_talebi_sayisi=np.clip(rng.poisson(2,n),0,9)
aylik_kullanim_saati=np.clip(rng.normal(42,18,n),3,120).round(1)
sehir=rng.choice(["Ankara","İstanbul","İzmir","Bursa","Antalya"],n)
uyelik_tipi=rng.choice(["Basic","Standard","Premium"],n,p=[.42,.38,.20])
skor=-2+.7*(destek_talebi_sayisi>=3)+1.1*(destek_talebi_sayisi>=5)+1.3*(abonelik_suresi<12)+.9*(uyelik_tipi=="Basic")-.8*(uyelik_tipi=="Premium")+.7*(gelir<35000)+.7*(aylik_kullanim_saati<20)
churn=rng.binomial(1,1/(1+np.exp(-skor)))
df=pd.DataFrame({"yas":yas,"gelir":gelir,"abonelik_suresi":abonelik_suresi,"destek_talebi_sayisi":destek_talebi_sayisi,"aylik_kullanim_saati":aylik_kullanim_saati,"sehir":sehir,"uyelik_tipi":uyelik_tipi,"churn":churn})
for col,oran in {"gelir":.05,"aylik_kullanim_saati":.03,"sehir":.04}.items(): df.loc[rng.choice(df.index,int(n*oran),replace=False),col]=np.nan

## 3. DataFrame ve CSV kaydetme

In [ ]:
df.to_csv("musteri_ayrilma_verisi.csv",index=False,encoding="utf-8-sig"); print(df.shape)

## 4. CSV dosyasını okuma

In [ ]:
df=pd.read_csv("musteri_ayrilma_verisi.csv")

## 5. İlk satırlar

In [ ]:
df.head()

## 6. Satır-sütun sayısı

In [ ]:
print("Satır:",df.shape[0],"Sütun:",df.shape[1])

## 7. Hedef değişken dağılımı

In [ ]:
print(df["churn"].value_counts()); print(df["churn"].value_counts(normalize=True).round(3))

## 8. Eksik değer kontrolü

In [ ]:
print(df.isnull().sum())

## 9. Öznitelik üretme

In [ ]:
df["gelir_grubu"]=pd.cut(df["gelir"],[0,35000,70000,np.inf],labels=["Düşük","Orta","Yüksek"]).astype(object)
df["destek_talebi_var_mi"]=np.where(df["destek_talebi_sayisi"]>0,"Evet","Hayır")
df["abonelik_yili"]=(df["abonelik_suresi"]/12).round(1)
df[["gelir_grubu","destek_talebi_var_mi","abonelik_yili"]].head()

## 10. X ve y hazırlama

In [ ]:
X=df.drop(columns="churn"); y=df["churn"]

## 11. Train-validation-test bölme

In [ ]:
X_train,X_tmp,y_train,y_tmp=train_test_split(X,y,test_size=.30,random_state=42,stratify=y)
X_val,X_test,y_val,y_test=train_test_split(X_tmp,y_tmp,test_size=.50,random_state=42,stratify=y_tmp)
print(X_train.shape,X_val.shape,X_test.shape)

## 12. Sayısal ve kategorik sütunlar

In [ ]:
sayisal=X.select_dtypes(include=np.number).columns.tolist(); kategorik=X.select_dtypes(exclude=np.number).columns.tolist(); print(sayisal); print(kategorik)

## 13. Eksik değer doldurma, ölçekleme ve One-Hot Encoding

In [ ]:
num=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
cat=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))])
pre=ColumnTransformer([("num",num,sayisal),("cat",cat,kategorik)])

## 14. Modelleri tanımlama

In [ ]:
modeller={"Logistic Regression":LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42),"KNN":KNeighborsClassifier(n_neighbors=9),"Decision Tree":DecisionTreeClassifier(max_depth=4,class_weight="balanced",random_state=42)}

## 15. Modelleri eğitme ve validation metrikleri

In [ ]:
sonuclar=[]; egitilen={}
for ad,model in modeller.items():
    p=Pipeline([("pre",pre),("model",model)]); p.fit(X_train,y_train); tah=p.predict(X_val)
    sonuclar.append({"model":ad,"accuracy":accuracy_score(y_val,tah),"precision":precision_score(y_val,tah,zero_division=0),"recall":recall_score(y_val,tah,zero_division=0),"f1":f1_score(y_val,tah,zero_division=0)}); egitilen[ad]=p

## 16. Validation sonuçlarını karşılaştırma

In [ ]:
val_df=pd.DataFrame(sonuclar).set_index("model").sort_values("f1",ascending=False); val_df.round(3)

## 17. En iyi modeli seçme

In [ ]:
en_iyi=val_df.index[0]; print("En iyi model:",en_iyi)
X_final=pd.concat([X_train,X_val]); y_final=pd.concat([y_train,y_val]); final_model=clone(egitilen[en_iyi]); final_model.fit(X_final,y_final)

## 18. Test metrikleri

In [ ]:
test_tahmin=final_model.predict(X_test)
metrikler={"accuracy":accuracy_score(y_test,test_tahmin),"precision":precision_score(y_test,test_tahmin,zero_division=0),"recall":recall_score(y_test,test_tahmin,zero_division=0),"f1":f1_score(y_test,test_tahmin,zero_division=0)}
print(metrikler)

## 19. Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,test_tahmin); print(cm); ConfusionMatrixDisplay(cm,display_labels=["Kalır (0)","Ayrılır (1)"]).plot(); plt.show()

## 20. Kısa sonuç yorumu

In [ ]:
print(f"Validation F1-score değerine göre en iyi model {en_iyi} oldu. Test F1-score: {metrikler['f1']:.3f}. Sentetik veride churn eşik tabanlı ilişkilerle üretildiği için bu ilişkileri daha iyi yakalayan model daha başarılı olmuştur.")